# Faster R-CNN — Inference

Loads the best trained Faster R-CNN model and runs detection on a test image, drawing bounding boxes with labels and scores.

**Requirements:**
- Best model weights: `best_fasterrcnn.pth`
- PyTorch, torchvision, matplotlib

In [ ]:
import torch
import torchvision
from PIL import Image
from torchvision.transforms import functional as F
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import matplotlib.pyplot as plt
import matplotlib.patches as patches

## Configuration

In [ ]:
WEIGHTS_PATH = "best_fasterrcnn.pth"
TEST_IMAGE_PATH = "../../data/unified/combined/test/images"
NUM_CLASSES = 13  # 12 unified classes + 1 background
CONFIDENCE_THRESHOLD = 0.5

CLASS_NAMES = [
    "Button", "Text", "Image", "Icon", "Input", "Link",
    "Checkbox", "Toggle", "Toolbar", "Navigation", "Modal", "Tab",
]

CLASS_COLOURS = [
    "#e6194b", "#3cb44b", "#ffe119", "#4363d8", "#f58231", "#911eb4",
    "#42d4f4", "#f032e6", "#bfef45", "#fabed4", "#469990", "#dcbeff",
]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Load Model

In [ ]:
def load_model(weights_path, num_classes, device):
    """Loads Faster R-CNN and restores trained weights."""
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    model.load_state_dict(torch.load(weights_path, map_location=device))
    model.to(device)
    model.eval()
    print(f"Model loaded from: {weights_path}")
    return model

## Run Inference

In [ ]:
def run_inference(model, image_path, device, confidence_threshold=0.5):
    """Runs inference on a single image and filters by confidence."""
    image = Image.open(image_path).convert("RGB")
    image_tensor = F.to_tensor(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(image_tensor)[0]

    mask   = outputs["scores"] >= confidence_threshold
    boxes  = outputs["boxes"][mask].cpu()
    labels = outputs["labels"][mask].cpu()
    scores = outputs["scores"][mask].cpu()

    return image, boxes, labels, scores

## Visualisation

In [ ]:
def visualise_detections(image, boxes, labels, scores, class_names,
                         class_colours, output_path=None):
    """Draws bounding boxes with labels and scores on the image."""
    fig, ax = plt.subplots(1, figsize=(14, 10))
    ax.imshow(image)

    for box, label, score in zip(boxes, labels, scores):
        x1, y1, x2, y2 = box.tolist()
        class_id = int(label)
        idx = class_id - 1
        class_name = class_names[idx] if 0 <= idx < len(class_names) else f"cls_{class_id}"
        colour = class_colours[idx % len(class_colours)]

        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor=colour, facecolor="none",
        )
        ax.add_patch(rect)

        label_text = f"{class_name}: {score:.0%}"
        ax.text(
            x1, y1 - 5, label_text, fontsize=9, fontweight="bold",
            color="white",
            bbox=dict(boxstyle="round,pad=0.2", facecolor=colour, alpha=0.8),
        )

    ax.set_title(
        f"Faster R-CNN Detections — {len(boxes)} objects (threshold: {CONFIDENCE_THRESHOLD})",
        fontsize=14, fontweight="bold",
    )
    ax.axis("off")
    fig.tight_layout()

    if output_path:
        fig.savefig(output_path, dpi=150)
        print(f"  Result saved to: {output_path}")

    plt.show()
    plt.close(fig)

## Run

In [ ]:
import glob

print("  Faster R-CNN — Inference on Test Image")

model = load_model(WEIGHTS_PATH, NUM_CLASSES, DEVICE)

# Pick the first test image if TEST_IMAGE_PATH is a directory
image_path = TEST_IMAGE_PATH
if os.path.isdir(image_path):
    extensions = ("*.png", "*.jpg", "*.jpeg", "*.webp")
    images = []
    for ext in extensions:
        images.extend(glob.glob(os.path.join(image_path, ext)))
    if not images:
        raise FileNotFoundError(f"No images found in {image_path}")
    image_path = sorted(images)[0]
    print(f"  Using first test image: {os.path.basename(image_path)}")

image, boxes, labels, scores = run_inference(
    model, image_path, DEVICE, CONFIDENCE_THRESHOLD,
)

print(f"\n Detections in: {image_path}")
print(f"Total: {len(boxes)} objects (confidence >= {CONFIDENCE_THRESHOLD})\n")

for i, (box, label, score) in enumerate(zip(boxes, labels, scores)):
    idx = int(label) - 1
    class_name = CLASS_NAMES[idx] if 0 <= idx < len(CLASS_NAMES) else f"cls_{int(label)}"
    coords = box.tolist()
    print(f" {i+1}. {class_name:<15} confidence: {score:.0%}  "
          f"bbox: [{coords[0]:.0f}, {coords[1]:.0f}, "
          f"{coords[2]:.0f}, {coords[3]:.0f}]")

visualise_detections(
    image, boxes, labels, scores,
    CLASS_NAMES, CLASS_COLOURS,
    output_path="../../results/faster-rcnn/inference_result.png",
)